# Preprocesamiento y Limpieza de Datos

El objetivo es transformar los archivos JSON en CSV limpios para analisis posterior.


## Importar librerías


In [14]:
import pandas as pd
import json
import os
import glob

## Exploración

Explorar estructura de un JSON:


In [2]:
# Leer un archivo JSON
archivo = '../data/raw/operaciones_semana_2025-06-01_2025-06-06.json'

with open(archivo, 'r', encoding='utf-8') as f:
    datos = json.load(f)

print(f"Archivo leido: {archivo}")
print(f"Claves principales: {list(datos.keys())}")


Archivo leido: ../data/raw/operaciones_semana_2025-06-01_2025-06-06.json
Claves principales: ['success', 'message', 'result']


### Extraer las operaciones

Las operaciones estan dentro de datos['result']['operaciones']


In [3]:
# Extraer las operaciones
operaciones = datos['result']['operaciones']

print(f"Total de operaciones: {len(operaciones)}")
print(f"Tipo de dato: {type(operaciones)}")


Total de operaciones: 720
Tipo de dato: <class 'list'>


### Convertir a DataFrame

Convertir la lista de operaciones a un DataFrame de pandas


In [4]:
# Convertir lista de diccionarios a DataFrame
df = pd.DataFrame(operaciones)

print(f"Dimensiones del DataFrame: {df.shape}")
print(f"Columnas: {df.columns.tolist()}")


Dimensiones del DataFrame: (720, 22)
Columnas: ['fechaOperacion', 'fechaConcertacion', 'tipoOperacion', 'tipoModalidad', 'tipoOiv', 'grano', 'volumenTN', 'condicionCalidad', 'condicionCalidadAuxiliar', 'procedenciaProvincia', 'procedenciaLocalidad', 'simboloPrecioPorTN', 'precioTN', 'lugarEntrega', 'fechaEntregaDesde', 'fechaEntregaHasta', 'condicionPago', 'esDestinoFinal', 'cosecha', 'numeroOperacionInstancia', 'nroOperacion', 'esUltimaInstancia']


### Ver las primeras filas

In [11]:
# Ver las primeras 3 filas
df.head(3)


,fechaOperacion,fechaConcertacion,tipoOperacion,tipoModalidad,tipoOiv,grano,volumenTN,condicionCalidad,condicionCalidadAuxiliar,procedenciaProvincia,...,precioTN,lugarEntrega,fechaEntregaDesde,fechaEntregaHasta,condicionPago,esDestinoFinal,cosecha,numeroOperacionInstancia,nroOperacion,esUltimaInstancia
0,2025-06-03T12:59:42.993,2025-05-09T00:00:00,Fijación,Compraventa,Fijar Precio/Prec. Cam.,MAIZ,250.00,Cámara,,SANTA FE,...,205000.00,Rosario S/En destino,2025-05-09T00:00:00,2025-05-20T00:00:00,A plazo,SI,COSECHA 24/25,3,2118370255,Si
1,2025-06-03T12:59:42.853,2025-02-25T00:00:00,Fijación,Compraventa,Fijar Precio/Mercado Comprador,SOJA,1500.00,Fábrica,,SANTA FE,...,325000.00,Rosario N/En destino,2025-04-01T00:00:00,2025-04-30T00:00:00,A plazo,SI,COSECHA 24/25,4,906230259,Si
2,2025-06-03T12:59:42.697,2025-01-30T00:00:00,Fijación,Compraventa,Fijar Precio/Prec. Cam.,TRIGO PAN,49.00,Art. 12,,CÓRDOBA,...,235800.00,Zona 9/En destino,2025-01-30T00:00:00,2025-02-28T00:00:00,A plazo,SI,COSECHA 24/25,10,540960259,Si


## Procesar todos los archivos JSON y exportar como CSV

In [15]:
# Obtener todos los archivos JSON de raw/
archivos_json = glob.glob('../data/raw/*.json')

print(f"Archivos JSON encontrados: {len(archivos_json)}\n")

# Procesar cada archivo
for archivo in archivos_json:
    # Leer JSON
    with open(archivo, 'r', encoding='utf-8') as f:
        datos = json.load(f)
    
    # Convertir a DataFrame
    operaciones = datos['result']['operaciones']
    df = pd.DataFrame(operaciones)
    
    # Crear nombre del CSV
    nombre_base = os.path.basename(archivo).replace('.json', '.csv')
    ruta_csv = f'../data/staging/{nombre_base}'
    
    # Exportar
    df.to_csv(ruta_csv, index=False, encoding='utf-8')
    
    print(f"{nombre_base}: {len(df)} operaciones")


Archivos JSON encontrados: 6

operaciones_semana_2025-05-11_2025-05-17.csv: 2164 operaciones
operaciones_semana_2025-05-25_2025-05-31.csv: 2243 operaciones
operaciones_semana_2025-06-01_2025-06-06.csv: 720 operaciones
operaciones_semana_2025-04-27_2025-05-03.csv: 1478 operaciones
operaciones_semana_2025-05-04_2025-05-10.csv: 2144 operaciones
operaciones_semana_2025-05-18_2025-05-24.csv: 2261 operaciones


In [1]:
import pandas as pd

Al investigar los valores históricos del dólar en Ámbito, pudimos hacer un GET desde Postman para extraer un JSON con los datos.https://mercados.ambito.com//dolar/oficial/grafico/2025-04-27/2025-06-04

In [15]:
# Leer json
df_dolar = pd.read_json('../data/raw/dolar_historico_granos.json')
df_dolar = df_dolar.T

# Reasignar nombres de columnas (la primera fila son los encabezados)
df_dolar.columns = df_dolar.iloc[0]
df_dolar = df_dolar.drop(df_dolar.index[0])

# Resetear el índice
df_dolar = df_dolar.reset_index(drop=True)

df_dolar

,fecha,28/04/2025,29/04/2025,30/04/2025,05/05/2025,06/05/2025,07/05/2025,08/05/2025,09/05/2025,12/05/2025,...,22/05/2025,23/05/2025,26/05/2025,27/05/2025,28/05/2025,29/05/2025,30/05/2025,02/06/2025,03/06/2025,04/06/2025
0,Dólar Oficial,1192.84,1194.45,1194.45,1216.47,1220.45,1152.76,1138.07,1166.96,1148.33,...,1161.91,1157.43,1163.94,1174.39,1176.88,1199.79,1209.35,1199.85,1203.75,1199.07


In [17]:
df_dolar.to_csv('../data/staging/dolar_historico_granos.csv')